## 00 — Prepare Azure ML workspace for corp-network access

One-time (or per-IP-change) setup so that the build/dev box on the corp network
can talk to the AML workspace's private-endpoint storage account, and so that
compute-cluster jobs can call `listKeys` on it despite the tenant's MCAPS
`StorageAccount_DisableLocalAuth_Modify` policy.

**Run this notebook when:**
- First time setting up this workspace from this machine.
- Your public IPv4 changed (ISP rotation, new office, VPN off/on).
- Jobs start failing with `AuthorizationFailure` (NSP) or
  `Key based authentication is not permitted` (MCAPS policy reverted shared-key) or
  `Identity of the specified managed compute … is not found` (cluster MI missing).

**Pre-flight checklist:**

| # | Step | Notes |
|---|------|-------|
| 1 | `az login --tenant <tenant>` | refresh if you see `AzureCliCredential` errors |
| 2 | **Turn off Global Secure Access (GSA) tunnel** | NSP is IPv4-only; GSA tunnels via Microsoft IPv6 → `AuthorizationFailure`. Verify: `netstat -rn \| grep ^default` should NOT show a `utun*` interface |
| 3 | Confirm the policy exemption exists (Azure Portal → Policy → Exemptions). See the *Policy exemption* md cell below. Without it, Step 2b reverts silently and jobs fail. |

**What this notebook does (idempotent — safe to re-run):**

1. Auth + workspace handle.
2. **Step 1** — Create / update NSP + profile + inbound `/24` rule for your current IP.
3. **Step 2** — Associate workspace storage with the NSP in `Enforced` mode.
4. **Step 2b** — Add subscription-scoped NSP rule for AML / ACR managed services + set `allowSharedKeyAccess=true` + `publicNetworkAccess=Enabled` on storage.
5. **Step 2c** — Sync the storage account's own `networkRuleSet.ipRules` to the same `/24` used by the NSP rule (this is what actually unblocks the AML Studio dataset-preview path when `publicNetworkAccess=Enabled`).
6. **Smoke-test** — reachability to the blob endpoint.
7. **Step 2d (optional)** — Provision the workspace managed VNet via SDK; set `INCLUDE_SPARK=True` to hydrate the Serverless Spark image cache (only needed if you actually use Spark).
8. **Step 3** — Assign system-assigned MI to the compute cluster + grant it `Storage Blob Data Contributor` on workspace storage. Required for jobs with `identity=UserIdentityConfiguration()`.

When the smoke test prints `HTTP 200/400 — reachable ✅` AND Step 3 prints
`✅ MI already system-assigned` + `✅ Role … already assigned`, the workspace
is ready and you can move on to `01_aml_cc_create_container_image.ipynb`.

### Troubleshooting

If the first cell that builds `MLClient` fails:
```
AzureCliCredential: Please run 'az login' to set up an account and login to the tenant
```
relogin from a shell:
```bash
az logout && az account clear
az login --tenant 00000000-0000-0000-0000-000000000000
```

If a later notebook's job fails with `Key based authentication is not permitted`
→ the policy exemption is missing or got reverted; re-run Step 2b and check
Azure Portal → Policy → Exemptions.

If a job fails with `Identity of the specified managed compute … is not found`
→ re-run Step 3 (cluster lost its MI, or you're on a fresh cluster).

In [8]:
import azure.ai.ml as aml
print(f"Azure ML SDK version: {aml.__version__}")

Azure ML SDK version: 1.33.0


In [9]:
# get a handle to the workspace
import os
from azure.ai.ml import MLClient
from dotenv import load_dotenv

config_file_name = "germanywest.env"
config_file_path = os.path.join(".", "config", config_file_name)
load_dotenv(dotenv_path=config_file_path, override=True)

from utils.amlauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

ml_client = MLClient(
    credential, settings.subscription_id, settings.resource_group, settings.workspace
)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [10]:
# get the location of the current workspace
workspace_location = ml_client.workspaces.get(settings.workspace).location
print(f"Workspace location: {workspace_location}")

Workspace location: germanywestcentral


### Wrap the storage account in a Network Security Perimeter (NSP)

The workspace's default storage account sits behind a managed private endpoint,
so its public `networkAcls` won't let traffic from this laptop reach the blob
data plane. The supported pattern (per
[NSP for Azure Storage](https://learn.microsoft.com/azure/storage/files/files-network-security-perimeter))
is to put the storage account into an **NSP** with an inbound access rule for
the client IP. Once associated in **Enforced** mode, NSP becomes the
authoritative inbound gate — the storage account's own firewall rules are no
longer evaluated, so we don't need to touch `networkAcls`.

NSPs are **regional**, so we create one in the same region as the workspace
storage (e.g. `germanywestcentral`).

The next three cells (idempotent — safe to re-run):

1. **Create / update NSP + profile + inbound IP rule**
   - NSP: `aml-build-clients-{region}`
   - Profile: `clients`
   - Rule: `client-{my_ip_dashed}` allowing the current public IPv4.
   - Stale `client-*` rules from previous IPs are removed.
2. **Associate the workspace storage account** with that profile in `Enforced`
   mode (async LRO — waits for completion).
3. **Smoke test** — poll the blob endpoint until NSP enforcement propagates
   (~60–120 s).

> **Note on Global Secure Access (GSA):** when GSA is enabled, your Azure
> egress is tunneled over IPv6 from a Microsoft-owned address. NSP currently
> only supports IPv4 rules, so storage will return `AuthorizationFailure`.
> Turn GSA off for this workflow.

In [11]:
# Step 1: create (or update) an NSP + profile + inbound access rule covering
# THIS client's current public IPv4 address (or a configurable CIDR range
# around it, so an ISP-issued IP change inside the same block doesn't require
# re-running this cell).
#
# History behaviour is controlled by PRUNE_OLD_CLIENT_RULES below:
#   False (default) → old `client-*` rules from other networks (home, office,
#                     coffee shop) are LEFT IN PLACE so you keep access from
#                     them too. Recommended day-to-day.
#   True            → old `client-*` rules that don't match the current rule
#                     name are DELETED. Useful for cleanup or when you want
#                     strict "only this network has access" semantics.
#
# The current rule is created only if no existing rule with the same name
# already covers the same CIDR (idempotent — true no-op if nothing changed).
#
# Resources (in the same region as the workspace storage):
#   - NSP      : aml-build-clients-{region}
#   - Profile  : clients
#   - Rule     : client-{network_dashed}-{prefix}  e.g. client-{first}-x-x-x-24
#
# Tuneable:
#   CLIENT_CIDR_PREFIX = 32  → strict /32 (single IP)
#                        24  → cover the /24 around the current IP (recommended
#                              when your ISP rotates the last octet)
#                        16  → /16 (much larger; only if you really need it)
#
# Idempotent. Uses `credential` + `settings` from the AuthHelper cell.
# All identifying values (subscription id, IPv4) are partially masked in print
# output via `utils.redact` so the notebook is screenshot-safe.
#
# Caveat — Global Secure Access (GSA):
#   When GSA is enabled, traffic to Azure may egress over IPv6 from a
#   Microsoft-owned address. NSP currently supports IPv4 access rules only,
#   so storage will reject GSA-tunneled requests as `AuthorizationFailure`.
#   Turn GSA off before running the upload.

import ipaddress
import re
import requests
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import (
    NetworkSecurityPerimeter,
    NspProfile,
    NspAccessRule,
)
from utils.redact import mask_ip, mask_ip_in_name, redact_text

# How tight should the inbound rule be? /32 = just my IP, /24 = my local block.
CLIENT_CIDR_PREFIX = 24

# Set True to DELETE any `client-*` rule that doesn't match the rule for the
# current IP/CIDR. Leaves non-`client-*` rules (e.g. `allow-aml-subscription`)
# untouched. Default False = preserve history of every network you've used.
PRUNE_OLD_CLIENT_RULES = False

IPV4_RE = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")

def _get_client_ipv4() -> str:
    """Return this client's apparent public IPv4. Tries a couple of services."""
    for url in ("https://api.ipify.org", "https://ipv4.icanhazip.com"):
        try:
            ip = requests.get(url, timeout=5).text.strip()
            if IPV4_RE.match(ip):
                return ip
        except Exception:
            pass
    raise RuntimeError(
        "Could not determine an IPv4 public address. "
        "Disable Global Secure Access if it is forcing IPv6 egress."
    )

# Resolve storage account (need region — NSPs are regional)
ws = ml_client.workspaces.get(settings.workspace)
sa_name = ws.storage_account.rsplit("/", 1)[-1]
storage_mgmt = StorageManagementClient(credential, settings.subscription_id)
sa_props = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
sa_region = sa_props.location
sa_arm_id = sa_props.id
print(f"Storage account : {sa_name}  (region={sa_region})")

# Current public IPv4 → CIDR
my_ip = _get_client_ipv4()
my_cidr = ipaddress.ip_network(f"{my_ip}/{CLIENT_CIDR_PREFIX}", strict=False)
network_addr = str(my_cidr.network_address)
target_cidr = str(my_cidr)
print(f"Client IPv4     : {mask_ip(my_ip)}")
print(f"Rule CIDR       : {mask_ip(str(my_cidr.network_address))}/{my_cidr.prefixlen}  ({my_cidr.num_addresses} addresses)")
print(f"Prune mode      : {PRUNE_OLD_CLIENT_RULES}")

# Network management client
net_client = NetworkManagementClient(credential, settings.subscription_id)

# Stable names (include prefix so /24 vs /32 vs new block don't collide)
nsp_name     = f"aml-build-clients-{sa_region}"
profile_name = "clients"
rule_name    = f"client-{network_addr.replace('.', '-')}-{CLIENT_CIDR_PREFIX}"

# 1a. NSP (create or update)
nsp = net_client.network_security_perimeters.create_or_update(
    settings.resource_group, nsp_name,
    NetworkSecurityPerimeter(location=sa_region, tags={"app": "aml-build", "owner": "notebook"}),
)
print(f"✅ NSP        : {nsp.name}")

# 1b. Profile (create or update)
profile = net_client.network_security_perimeter_profiles.create_or_update(
    settings.resource_group, nsp_name, profile_name,
    NspProfile(),
)
print(f"✅ Profile    : {profile.name}")

# 1c. List existing rules — show what's already there (redacted output)
existing_rules = list(
    net_client.network_security_perimeter_access_rules.list(
        settings.resource_group, nsp_name, profile_name
    )
)
print(f"\nExisting access rules in profile '{profile_name}' ({len(existing_rules)}):")
for r in existing_rules:
    if r.address_prefixes:
        scope = "prefixes=[" + ", ".join(mask_ip(p.split("/")[0]) + (("/" + p.split("/")[1]) if "/" in p else "") for p in r.address_prefixes) + "]"
    elif getattr(r, "subscriptions", None):
        scope = "subscriptions=[<redacted>]"
    elif getattr(r, "service_tags", None):
        scope = "service_tags=[" + ", ".join(r.service_tags) + "]"
    else:
        scope = "scope=<empty>"
    print(f"   • {mask_ip_in_name(r.name):35s}  direction={r.direction}  {scope}")

# 1d. Optional pruning of stale client-* rules (only when explicitly requested)
if PRUNE_OLD_CLIENT_RULES:
    pruned = 0
    for r in existing_rules:
        if r.name and r.name.startswith("client-") and r.name != rule_name:
            net_client.network_security_perimeter_access_rules.delete(
                settings.resource_group, nsp_name, profile_name, r.name
            )
            print(redact_text(f"🗑️  Pruned stale client rule: {r.name}  prefixes={r.address_prefixes}"))
            pruned += 1
    if pruned == 0:
        print("\nNo stale client-* rules to prune.")
    # Refresh local list so the "match" check below sees the post-prune state
    existing_rules = [r for r in existing_rules
                      if not (r.name and r.name.startswith("client-") and r.name != rule_name)]

# 1e. Decide whether to create / update the rule for the current CIDR.
#     - If a rule with the same name AND same single prefix already exists,
#       it's a true no-op (skip the API call).
#     - If a rule with the same name exists but with different prefixes,
#       update it to the new CIDR (in case CLIENT_CIDR_PREFIX changed).
#     - Otherwise, create a new rule. Other `client-*` rules are left alone
#       unless PRUNE_OLD_CLIENT_RULES=True (above).
match = next((r for r in existing_rules if r.name == rule_name), None)
if match and match.address_prefixes == [target_cidr] and (match.direction or "").lower() == "inbound":
    print(redact_text(f"\n✅ Access rule already exists & matches — no change: {rule_name}  prefixes={match.address_prefixes}"))
else:
    action = "Updating" if match else "Creating"
    print(redact_text(f"\n{action} access rule: {rule_name}  direction=Inbound  prefixes=[{target_cidr}]"))
    rule = net_client.network_security_perimeter_access_rules.create_or_update(
        settings.resource_group, nsp_name, profile_name, rule_name,
        NspAccessRule(direction="Inbound", address_prefixes=[target_cidr]),
    )
    print(redact_text(f"✅ Access rule: {rule.name}  direction={rule.direction}  prefixes={rule.address_prefixes}"))

Storage account : amlwwywdos1036066399  (region=germanywestcentral)
Client IPv4     : 167.x.x.x
Rule CIDR       : 167.x.x.x/24  (256 addresses)
Prune mode      : False
✅ NSP        : aml-build-clients-germanywestcentral
✅ Profile    : clients

Existing access rules in profile 'clients' (3):
   • client-83-x-x-x-24                   direction=Inbound  prefixes=[83.x.x.x/24]
   • allow-aml-subscription               direction=Inbound  subscriptions=[<redacted>]
   • client-167-x-x-x-24                  direction=Inbound  prefixes=[167.x.x.x/24]

✅ Access rule already exists & matches — no change: client-167-x-x-x-24  prefixes=['167.x.x.x/24']


In [12]:
# Step 2: associate the workspace storage account with the NSP profile.
#
# This is an async (LRO) operation. Once it returns "Succeeded", NSP enforces
# the inbound IP rule on the storage account's data plane (blob, file, etc.).
#
# Idempotent + skip-if-unchanged: if an association with the same name already
# points the same storage account at the same profile in `Enforced` mode, the
# LRO is skipped entirely (no API call). Otherwise create/update it.

from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.network.models import NspAssociation, SubResource

assoc_name = f"{sa_name}-to-{profile_name}"
TARGET_ACCESS_MODE = "Enforced"   # use "Learning" first if you want to audit before enforcing

# Check current association state (if any)
try:
    current_assoc = net_client.network_security_perimeter_associations.get(
        settings.resource_group, nsp_name, assoc_name
    )
except ResourceNotFoundError:
    current_assoc = None

def _id_equal(a, b):
    return (a or "").lower() == (b or "").lower()

is_match = (
    current_assoc is not None
    and _id_equal(getattr(current_assoc.private_link_resource, "id", None), sa_arm_id)
    and _id_equal(getattr(current_assoc.profile, "id", None), profile.id)
    and (current_assoc.access_mode or "").lower() == TARGET_ACCESS_MODE.lower()
    and (current_assoc.provisioning_state or "").lower() == "succeeded"
)

if is_match:
    print(f"✅ Association already exists & matches — no change: {current_assoc.name}")
    print(f"   accessMode  : {current_assoc.access_mode}")
    print(f"   state       : {current_assoc.provisioning_state}")
else:
    action = "Updating" if current_assoc else "Creating"
    print(f"{action} association: storage {sa_name} → NSP {nsp_name}/{profile_name} (accessMode={TARGET_ACCESS_MODE}) ...")
    assoc_poller = net_client.network_security_perimeter_associations.begin_create_or_update(
        settings.resource_group, nsp_name, assoc_name,
        NspAssociation(
            private_link_resource=SubResource(id=sa_arm_id),
            profile=SubResource(id=profile.id),
            access_mode=TARGET_ACCESS_MODE,
        ),
    )
    assoc = assoc_poller.result()
    print(f"✅ Association : {assoc.name}")
    print(f"   accessMode  : {assoc.access_mode}")
    print(f"   state       : {assoc.provisioning_state}")
    print("\nWait ~60–120s for NSP enforcement to propagate, then re-run the smoke-test cell.")

✅ Association already exists & matches — no change: amlwwywdos1036066399-to-clients
   accessMode  : Enforced
   state       : Succeeded


### Policy exemption — required so shared-key stays enabled

In Microsoft-internal tenants the management-group policy assignment
`MCAPSGovDeployPolicies` (Tenant Root) includes the
`StorageAccount_DisableLocalAuth_Modify` policy with a `modify` effect that
auto-flips `allowSharedKeyAccess` back to `false` on every storage account in
the tenant. Without an exemption, the cell below will appear to succeed but the
value will silently revert, and AML compute-cluster jobs will fail with:

```
Key based authentication is not permitted on this storage account
ErrorCode: ForbiddenError
```

**Exemption created for this workspace storage** (one-time, done from the
Azure Portal as tenant admin since RG-level Owner is not enough):

| Field | Value |
|---|---|
| Policy assignment | `MCAPSGovDeployPolicies` (scope: Tenant Root MG `787eb5ff-…`) |
| Policy reference | `StorageAccount_DisableLocalAuth_Modify` |
| Exemption scope | `/subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.Storage/storageAccounts/{sa_name}` |
| Category | `Mitigated` |
| Justification | AML v2 Execution service performs `listKeys` on the workspace storage account during job orchestration on user-managed compute clusters. Mitigated by NSP IPv4 allow-list (cells above) + Entra RBAC for all OAuth data-plane access. |

CLI form (run as someone with `policyAssignments/exempt/action` on the MG):

```bash
az policy exemption create \
  --name "Allow-SharedKey-<sa-name>" \
  --display-name "AML compute-cluster needs listKeys on workspace storage" \
  --policy-assignment "/providers/Microsoft.Management/managementGroups/<tenant-mg>/providers/Microsoft.Authorization/policyAssignments/MCAPSGovDeployPolicies" \
  --policy-definition-reference-ids "StorageAccount_DisableLocalAuth_Modify" \
  --exemption-category "Mitigated" \
  --scope "/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<sa-name>"
```

Once the exemption exists, the next cell (`Step 2b`) can flip
`allowSharedKeyAccess=true` and have it stick.

In [13]:
# Step 2b: open the workspace storage for AML control plane.
#
# Once NSP is Enforced (cell above), three more things must be true so that the
# AML execution service, ACR Tasks, and your laptop can all do their jobs:
#
#   1. NSP profile must have an Inbound rule allowing the AML subscription
#      (covers AML/ACR managed services running inside your sub) — in addition
#      to the client/{ip} rule.
#   2. Storage `allowSharedKeyAccess=true` — AML v2 Execution service does
#      `listKeys` regardless of identity settings on the job. Without this,
#      jobs fail with "Key based authentication is not permitted on this
#      storage account" (ForbiddenError). Requires a Policy Exemption for
#      `StorageAccount_DisableLocalAuth_Modify` if MCAPS Gov is enforced.
#   3. Storage `publicNetworkAccess=Enabled` — NSP itself is the firewall now,
#      but the storage public endpoint must accept traffic from it.
#
# All three are idempotent; rerunning is a no-op when nothing needs to change.

from azure.mgmt.network.models import NspAccessRule, SubscriptionId
from azure.mgmt.storage.models import StorageAccountUpdateParameters
from utils.redact import redact_text

# 2b.1 — NSP inbound rule allowing AML services in this subscription
aml_rule_name = "allow-aml-subscription"
aml_rule = net_client.network_security_perimeter_access_rules.create_or_update(
    settings.resource_group, nsp_name, profile_name, aml_rule_name,
    NspAccessRule(
        direction="Inbound",
        subscriptions=[SubscriptionId(id=f"/subscriptions/{settings.subscription_id}")],
    ),
)
print(redact_text(
    f"✅ NSP rule  : {aml_rule.name}  direction={aml_rule.direction}  "
    f"subs={[s.id for s in (aml_rule.subscriptions or [])]}"
))

# 2b.2 — Storage shared-key + public network access
#
# `public_network_access` is an enum (PublicNetworkAccess.ENABLED / .DISABLED),
# whose str() is "PublicNetworkAccess.ENABLED". Compare by `.value` (or just
# coerce via str(...).endswith()) so we don't PATCH every run.
sa_state = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)

def _pna_is_enabled(v) -> bool:
    if v is None:
        return False
    # enum → "Enabled" / "Disabled" via .value; str fallback handles plain strings
    raw = getattr(v, "value", v)
    return str(raw).lower() == "enabled"

shared_key_on = sa_state.allow_shared_key_access is True
pna_on        = _pna_is_enabled(sa_state.public_network_access)
print(f"   storage now: shared_key={sa_state.allow_shared_key_access}  public_network={sa_state.public_network_access}")

patch = {}
if not shared_key_on:
    patch["allow_shared_key_access"] = True
if not pna_on:
    patch["public_network_access"] = "Enabled"

if patch:
    print(f"Applying storage PATCH : {patch}")
    storage_mgmt.storage_accounts.update(
        settings.resource_group, sa_name,
        StorageAccountUpdateParameters(**patch),
    )
    after = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
    print(f"✅ storage now: shared_key={after.allow_shared_key_access}  public_network={after.public_network_access}")
    if after.allow_shared_key_access is not True:
        print("\n⚠️  shared-key still False — MCAPS Gov policy "
              "`StorageAccount_DisableLocalAuth_Modify` is reverting it. "
              "Create a Policy Exemption (Mitigated) for this storage account "
              "(see markdown cell above).")
else:
    print("✅ No storage change needed.")

✅ NSP rule  : allow-aml-subscription  direction=Inbound  subs=['/subscriptions/6753…8abe']
   storage now: shared_key=True  public_network=PublicNetworkAccess.ENABLED
✅ No storage change needed.


### Step 2c — Sync the storage account firewall to the same /24

> **Why this step exists — NSP vs storage firewall:**
> Setting `publicNetworkAccess=Enabled` (Step 2b) keeps the storage account's
> **own** `networkRuleSet` as the authoritative inbound firewall for the public
> blob/file endpoints. In this mode the **NSP is effectively bypassed for
> direct data-plane traffic** — NSP rules only apply to the resources NSP
> explicitly gates (and to subscription-scoped service-to-service calls). The
> AML Studio dataset-preview path hits the public blob endpoint directly, so
> the storage `ipRules` list is what decides whether you get through.
>
> Only `publicNetworkAccess=SecuredByPerimeter` would make NSP the sole gate
> and cause storage `networkRuleSet` to be ignored. We don't use that mode
> here because (a) it's still in preview for Storage in many regions and
> (b) Step 2b's `bypass=AzureServices` + shared-key requirement plays more
> reliably with `Enabled`.
>
> Net effect: Step 1 keeps NSP tidy (useful for any NSP-gated paths and as
> documentation of which networks you've used), but **this Step 2c is what
> actually unblocks the Studio preview and any laptop-direct blob access**.

Symptom when this is stale (your IP rotated, storage `ipRules` still lists
the old `/32`):

```
StreamError(PermissionDenied(... PrivateEndpointResolutionFailureException
was caused by DataAccessPrivateEndpointResolutionException.
Cannot authenticate data access to ... with Workspace system assigned identity.
Make sure to connect to the Workspace with a Private Endpoint or whitelist
your public ip address on storage. ...))
```

The next cell keeps storage `ipRules` in sync with the NSP rule:

1. Computes the same `/24` (or `CLIENT_CIDR_PREFIX`) used by Step 1.
2. Adds it to `networkRuleSet.ipRules` if missing.
3. Removes any **single-IP** `/32` entries inside that `/24` (the stale ones
   left over from previous ISP-assigned addresses). Other `/32` or `/24`
   ranges from different networks (home, VPN exit) are preserved unless
   `PRUNE_OTHER_STORAGE_IP_RULES = True`.

Idempotent — no-op if `ipRules` already contains the current `/24` exactly.


In [14]:
# Step 2c: keep storage account firewall ipRules in sync with the NSP /24 rule.
#
# IMPORTANT — why this is needed even though we have NSP:
#   With publicNetworkAccess=Enabled (set in Step 2b), the storage account's
#   OWN networkRuleSet is the authoritative inbound firewall for public
#   blob/file endpoints. NSP is effectively BYPASSED for direct data-plane
#   traffic in this mode — the storage firewall is evaluated first and a
#   stale entry here will block you regardless of how clean the NSP rules are.
#   Only publicNetworkAccess=SecuredByPerimeter would invert that (NSP gates,
#   storage networkRuleSet ignored), and we are deliberately not using that
#   mode (see the markdown cell above for why).
#
# Reuses values computed in Step 1 (sa_name, sa_region, my_cidr, target_cidr,
# CLIENT_CIDR_PREFIX) and the existing `storage_mgmt` client.
#
# Storage firewall semantics (when publicNetworkAccess = Enabled):
#   - defaultAction = Deny → only listed ipRules / vnetRules / resourceAccessRules
#     can reach the public endpoint.
#   - bypass = AzureServices → AML/ACR control plane still gets through.
#   - ipRules accept a single IPv4 or a CIDR range with /24..30 prefixes.
#     /32 is normalized to a bare IP by the API.
#
# Behaviour:
#   PRUNE_OTHER_STORAGE_IP_RULES = False (default)
#     - Add target /24 if not already present.
#     - Remove only bare-IP entries that fall *inside* the target /24
#       (stale ISP-assigned addresses from the same corp block).
#     - Leave unrelated /32 or CIDR rules alone (e.g. home network).
#   PRUNE_OTHER_STORAGE_IP_RULES = True
#     - Delete every existing ipRule that is not exactly the target /24.

import ipaddress as _ip
from azure.mgmt.storage.models import (
    StorageAccountUpdateParameters,
    NetworkRuleSet,
    IPRule,
    ResourceAccessRule,
    VirtualNetworkRule,
)
from utils.redact import mask_ip

PRUNE_OTHER_STORAGE_IP_RULES = False

# Re-read live state (Step 2b may have updated bypass/pna).
sa_state = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
nrs = sa_state.network_rule_set
current_rules = list(nrs.ip_rules or [])

print(f"Storage firewall before sync ({len(current_rules)} ipRules):")
for r in current_rules:
    print(f"   • {mask_ip(r.ip_address_or_range.split('/')[0])}"
          f"{('/' + r.ip_address_or_range.split('/')[1]) if '/' in r.ip_address_or_range else ''}")

target_net = _ip.ip_network(target_cidr, strict=False)

def _entry_to_net(entry: str) -> _ip.IPv4Network:
    """Normalize an ipRule value (bare IP or CIDR) to a network."""
    return _ip.ip_network(entry if "/" in entry else f"{entry}/32", strict=False)

kept = []
removed = []
already_has_target = False

for r in current_rules:
    val = r.ip_address_or_range
    net = _entry_to_net(val)

    if net == target_net:
        already_has_target = True
        kept.append(r)
        continue

    if PRUNE_OTHER_STORAGE_IP_RULES:
        removed.append(val)
        continue

    # Default: drop stale /32s inside the target /24, keep everything else.
    if net.prefixlen == 32 and net.subnet_of(target_net):
        removed.append(val)
    else:
        kept.append(r)

if not already_has_target:
    kept.append(IPRule(ip_address_or_range=target_cidr, action="Allow"))

# Decide if a PATCH is actually needed.
before_set = {r.ip_address_or_range for r in current_rules}
after_set  = {r.ip_address_or_range for r in kept}

if before_set == after_set:
    print("\n✅ Storage firewall already matches NSP /24 — no change.")
else:
    print(f"\nApplying storage firewall sync:")
    if removed:
        for v in removed:
            ip_part = v.split("/")[0]
            suffix  = ("/" + v.split("/")[1]) if "/" in v else ""
            print(f"   - removing  {mask_ip(ip_part)}{suffix}")
    if not already_has_target:
        ip_part = target_cidr.split("/")[0]
        print(f"   + adding    {mask_ip(ip_part)}/{target_net.prefixlen}")

    new_nrs = NetworkRuleSet(
        bypass=nrs.bypass or "AzureServices",
        default_action=nrs.default_action or "Deny",
        ip_rules=kept,
        virtual_network_rules=[
            VirtualNetworkRule(
                virtual_network_resource_id=v.virtual_network_resource_id,
                action=v.action,
                state=v.state,
            )
            for v in (nrs.virtual_network_rules or [])
        ],
        resource_access_rules=[
            ResourceAccessRule(tenant_id=r.tenant_id, resource_id=r.resource_id)
            for r in (nrs.resource_access_rules or [])
        ],
    )
    storage_mgmt.storage_accounts.update(
        settings.resource_group, sa_name,
        StorageAccountUpdateParameters(network_rule_set=new_nrs),
    )
    after = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
    print(f"\nStorage firewall after sync ({len(after.network_rule_set.ip_rules)} ipRules):")
    for r in after.network_rule_set.ip_rules:
        v = r.ip_address_or_range
        ip_part = v.split("/")[0]
        suffix  = ("/" + v.split("/")[1]) if "/" in v else ""
        print(f"   • {mask_ip(ip_part)}{suffix}")
    print("✅ Storage firewall sync complete.")


Storage firewall before sync (1 ipRules):
   • 167.x.x.x/24

✅ Storage firewall already matches NSP /24 — no change.


In [15]:
# Smoke test: can we now reach the blob endpoint? Poll up to 90s for the
# NSP enforcement to propagate. We just need a TCP+TLS handshake — any HTTP
# response means we're through (e.g. 400 InvalidQueryParameterValue is fine,
# storage replied). A connect timeout means NSP hasn't propagated yet.
import time as _t
import urllib.error
import urllib.request
import ssl

blob_url = f"https://{sa_name}.blob.core.windows.net/"
ctx = ssl.create_default_context()
ok = False
for i in range(9):
    try:
        with urllib.request.urlopen(blob_url, timeout=5, context=ctx) as r:
            print(f"[{i*10:3d}s] HTTP {r.status} — reachable ✅")
            ok = True
            break
    except urllib.error.HTTPError as e:
        print(f"[{i*10:3d}s] HTTP {e.code} — reachable ✅")
        ok = True
        break
    except Exception as e:
        print(f"[{i*10:3d}s] not yet ({type(e).__name__}: {e}) — waiting ...")
        _t.sleep(10)

if not ok:
    print("\n❌ Still not reachable after 90s — NSP may need more time, "
          "or GSA is forcing IPv6 egress. Disable GSA and retry.")

[  0s] HTTP 400 — reachable ✅


### Step 2d — Provision the workspace managed VNet (optional, Spark-enabled)

When the workspace has a managed VNet (`isolation_mode != disabled`), the
managed outbound resources (system private endpoints to storage / keyvault /
ACR) are **declared** at workspace-create time but only **materialised** on
first use. Two symptoms of an un-provisioned managed VNet:

- `managed_network.status.status` ≠ `Active`
- `managed_network.status.spark_ready` = `false` → serverless Spark sessions
  (and a few Studio data-preview paths that pre-warm Spark) will hang or fail
  with `PrivateEndpointResolutionFailureException` until the image cache is
  hydrated inside the managed VNet.

The next cell is idempotent:

1. Reads `managed_network.status` from the workspace.
2. If `status != Active` **or** `include_spark=True` and `spark_ready != True`,
   triggers `begin_provision_network(include_spark=True)` and waits for the LRO.
3. Otherwise it's a no-op.

> **Skip this step if you don't use Serverless Spark.** A non-Spark workflow
> only needs `status=Active` (which is normally already true after workspace
> creation). Setting `INCLUDE_SPARK = False` skips the (5–10 min) Spark image
> hydration but still ensures the managed VNet itself is Active.

> **Note:** the Studio Networking blade does not surface `spark_ready`; you
> can only see it via the SDK / CLI (`az ml workspace show ... --query
> managed_network.status`). There is no separate "Spark endpoint" resource —
> Spark reuses the existing `__SYS_PE_*` outbound rules with their
> `spark_enabled=true` flag.


In [ ]:
# Step 2d: provision the workspace managed VNet, optionally hydrating the
# Serverless Spark image cache.
#
# SDK call: ml_client.workspaces.begin_provision_network(
#               workspace_name=..., include_spark=True|False)
#   - When include_spark=True, AML spins up a transient compute inside the
#     managed VNet, pulls the Spark runtime container image through the
#     existing outbound rules, and caches it. Subsequent serverless Spark
#     sessions then start in ~30s instead of 5–10min.
#   - When include_spark=False, only the managed PE/FQDN outbound rules are
#     materialised; spark_ready stays False.
#
# Idempotent: skipped entirely when the workspace already reports
# `status=Active` (and `spark_ready=True` if include_spark was requested).

INCLUDE_SPARK = False   # flip to True only if you use Serverless Spark

ws = ml_client.workspaces.get(settings.workspace)
mn = getattr(ws, "managed_network", None)

if mn is None or (getattr(mn, "isolation_mode", "") or "").lower() in ("", "disabled"):
    print("ℹ️  Workspace has no managed VNet (isolation_mode=disabled) — nothing to provision.")
else:
    status_obj = getattr(mn, "status", None)
    cur_status = getattr(status_obj, "status", None)
    cur_spark  = getattr(status_obj, "spark_ready", None)
    print(f"Managed VNet isolation_mode : {mn.isolation_mode}")
    print(f"   status                   : {cur_status}")
    print(f"   spark_ready              : {cur_spark}")

    needs_provision = (
        (cur_status or "").lower() != "active"
        or (INCLUDE_SPARK and cur_spark is not True)
    )

    if not needs_provision:
        print("\n✅ Managed VNet already provisioned to required state — no change.")
    else:
        print(f"\nTriggering begin_provision_network(include_spark={INCLUDE_SPARK}) ...")
        poller = ml_client.workspaces.begin_provision_network(
            workspace_name=settings.workspace,
            include_spark=INCLUDE_SPARK,
        )
        # LRO can take 1–3 min without Spark, 5–10 min with Spark.
        result = poller.result()
        # Re-read for final state.
        ws_after = ml_client.workspaces.get(settings.workspace)
        s = ws_after.managed_network.status
        print(f"\n✅ Managed VNet provisioned")
        print(f"   status      : {s.status}")
        print(f"   spark_ready : {s.spark_ready}")


### Step 3 — Assign a managed identity to the compute cluster

Jobs submitted with `identity=UserIdentityConfiguration()` require the target
compute cluster to have a managed identity (system- or user-assigned). The
cluster MI is what AML uses to mount the workspace storage at job start and
to impersonate your Entra token at runtime.

Without an MI on the cluster, you'll see:

```
UserError: Identity of the specified managed compute … is not found.
Set up managed identity, refer "https://docs.microsoft.com/.../managed-identity".
```

The next cell (idempotent):

1. Assigns a **system-assigned** managed identity to the cluster
   (`CLUSTER_NAME` below — change if needed).
2. Grants the cluster MI **Storage Blob Data Contributor** on the workspace
   storage account so it can read/write `azureml-blobstore-*` over Entra.

If the cluster already has an MI with the right role, both steps are no-ops.

> **Required permission:** to assign the role, the principal running this
> notebook needs `Microsoft.Authorization/roleAssignments/write` on the
> storage account scope (e.g. **Owner** or **User Access Administrator** on
> the resource group / storage account). If you only have **Contributor**,
> the role assignment step will fail and a tenant admin must do that part
> manually (Portal → Storage account → Access control (IAM) → Add role
> assignment → Storage Blob Data Contributor → cluster MI principalId).

In [ ]:
# Step 3 — Assign system-assigned MI to the compute cluster + grant RBAC.
#
# Idempotent:
#   - If the cluster already has system_assigned identity, leaves it.
#   - If the role assignment already exists, leaves it.
#
# NOTE on the identity-assignment path:
#   `ml_client.compute.begin_create_or_update(cluster)` hits a known bug when
#   the cluster has `network_settings` set but no `subnet` ARM id — it raises:
#     AttributeError: 'NoneType' object has no attribute 'lower'
#   in azure.ai.ml._schema._utils.utils.validate_arm_str().
#   Workaround: shell out to `az ml compute update --identity-type SystemAssigned`,
#   which uses the dedicated PATCH endpoint and skips the broken validator.
#
# The RBAC step still uses the SDK (azure-mgmt-authorization). The role is
# resolved by display-name via `role_definitions.list` (filter syntax is the
# OData $filter spec, see https://learn.microsoft.com/rest/api/authorization/role-definitions/list)
# so we don't hard-code a GUID.

import subprocess
import uuid

from azure.core.exceptions import HttpResponseError, ResourceExistsError
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

CLUSTER_NAME = "Cluster-A100-1GPU"
ROLE_NAME    = "Storage Blob Data Contributor"

# 3a. Read current cluster, check identity.
cluster = ml_client.compute.get(CLUSTER_NAME)
current_identity_type = (getattr(cluster.identity, "type", "") or "").lower()
print(f"Cluster '{CLUSTER_NAME}'  identity_type={current_identity_type or '(none)'}")

if current_identity_type not in {"systemassigned", "system_assigned"}:
    print("Assigning system-assigned managed identity via az CLI ...")
    # `az ml compute update` requires the workspace coordinates explicitly.
    cmd = [
        "az", "ml", "compute", "update",
        "--name", CLUSTER_NAME,
        "--identity-type", "SystemAssigned",
        "--resource-group", settings.resource_group,
        "--workspace-name", settings.workspace,
        "--subscription", settings.subscription_id,
    ]
    print("  $", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            "az ml compute update failed. Make sure the Azure CLI ML extension "
            "is installed: az extension add -n ml"
        )
    # Re-fetch the cluster to pick up the new identity.
    cluster = ml_client.compute.get(CLUSTER_NAME)
    print(f"✅ Updated  identity_type={cluster.identity.type}")
else:
    print("✅ MI already system-assigned, no change.")

principal_id = cluster.identity.principal_id
print(f"   Cluster MI principalId : {principal_id}")

if not principal_id:
    raise RuntimeError(
        "Cluster identity.principal_id is empty — wait a few seconds for "
        "AAD propagation and re-run this cell."
    )

# 3b. Resolve the role definition by display-name (no magic GUID).
#     Use `api_version="2022-04-01"` so we get the modern RoleAssignment API
#     surface that accepts `principal_type` (older default rejected it →
#     "MalformedRoleAssignmentRequest").
auth_client = AuthorizationManagementClient(
    credential, settings.subscription_id, api_version="2022-04-01",
)

# Scope for the *role definition lookup* is the subscription; role definitions
# are inherited from there. We pass an OData filter so we only fetch the one.
sub_scope = f"/subscriptions/{settings.subscription_id}"
role_defs = list(auth_client.role_definitions.list(
    scope=sub_scope,
    filter=f"roleName eq '{ROLE_NAME}'",
))
if not role_defs:
    raise RuntimeError(f"Role '{ROLE_NAME}' not found at scope {sub_scope}.")
role_def = role_defs[0]
role_def_id = role_def.id          # full ARM path used in the assignment
role_def_guid = role_def.name      # just the GUID, used to dedupe
print(f"   Role '{ROLE_NAME}'  guid={role_def_guid}")

# 3c. Grant the cluster MI that role on the workspace storage.
#     Check if an equivalent assignment already exists at the storage scope.
existing = [
    ra for ra in auth_client.role_assignments.list_for_scope(sa_arm_id)
    if ra.principal_id == principal_id
    and ra.role_definition_id.lower().endswith(role_def_guid.lower())
]

if existing:
    print(f"✅ Role '{ROLE_NAME}' already assigned (assignment_id={existing[0].name})")
else:
    print(f"Creating role assignment: {ROLE_NAME} → cluster MI ...")
    try:
        # Use the typed model. Passing a raw dict to `parameters` previously
        # produced "MalformedRoleAssignmentRequest" because the keys were sent
        # as snake_case rather than the camelCase the REST API expects.
        ra = auth_client.role_assignments.create(
            scope=sa_arm_id,
            role_assignment_name=str(uuid.uuid4()),
            parameters=RoleAssignmentCreateParameters(
                role_definition_id=role_def_id,
                principal_id=principal_id,
                principal_type="ServicePrincipal",
            ),
        )
        print(f"✅ Created  assignment_id={ra.name}")
    except (HttpResponseError, ResourceExistsError) as e:
        # ResourceExistsError = AAD replication race (rare); HttpResponseError =
        # most likely 403 because the caller lacks Microsoft.Authorization/roleAssignments/write.
        print(f"\n❌ Could not create role assignment automatically: {e.message if hasattr(e, 'message') else e}")
        print("\nFall back to manual assignment (run as tenant admin or someone with")
        print("User Access Administrator on the storage account):")
        print()
        print(f"  az role assignment create \\")
        print(f"    --assignee-object-id {principal_id} \\")
        print(f"    --assignee-principal-type ServicePrincipal \\")
        print(f"    --role '{ROLE_NAME}' \\")
        print(f"    --scope {sa_arm_id}")

### Ready

If the smoke-test cell printed `HTTP 200/400 — reachable ✅` AND Step 3
finished with `✅ MI already system-assigned` (or freshly assigned) plus
`✅ Role … already assigned`, the workspace is fully prepared:

- NSP allows your IP + AML subscription.
- Storage has shared-key + public network access (per the policy exemption).
- Compute cluster has a system-assigned MI with `Storage Blob Data Contributor`.

Next: open `01_aml_cc_create_container_image.ipynb` to build the custom AML
environment and run the smoke training job.

### Troubleshooting — AML Studio "Data preview" fails (400 / AxiosError)

When you click **Data → your asset → Preview** and see one of these:

```
AxiosError: Request failed with status code 400
    at .../manualChunk_data-fetch-*.js
```

or

```
StreamError(PermissionDenied(... PrivateEndpointResolutionFailureException
was caused by DataAccessPrivateEndpointResolutionException.
Cannot authenticate data access ... with Workspace system assigned identity.
Make sure to connect to the Workspace with a Private Endpoint or whitelist
your public ip address on storage.))
```

…the **error message is misleading**. Both error texts are emitted by the
Studio frontend whenever its preview pipeline fails for *any* reason
(network, auth, **or unsupported file format**). The "whitelist your IP" /
"private endpoint" wording is just generic boilerplate.

**Before assuming it's infrastructure, run the smoke test** in
[`smoke/verify_dataset_access.py`](smoke/verify_dataset_access.py):

```bash
cd 02-compute-cluster
python smoke/verify_dataset_access.py <data-asset-name>
```

It checks four layers independently:

1. AAD token for `storage.azure.com`
2. `BlobServiceClient.list_containers` (firewall + RBAC)
3. `ml_client.data.get` (control plane / asset metadata)
4. Direct blob read of the asset's actual files (data plane)

If all four pass, **the data is accessible**. The 400 is a Studio UI bug,
not a configuration problem. Common UI-only failure modes:

| Symptom | Real cause | Fix |
|---|---|---|
| `400 AxiosError` on a `uri_folder` containing `*.arrow` (HuggingFace `datasets`) | Studio's tabular preview only handles CSV / Parquet / JSON / JSONL. Arrow → 400. | Ignore — your training jobs still read it via `datasets.load_from_disk`. |
| `400 AxiosError` on a `uri_folder` with `.bin` / `.pt` / `.safetensors` weights | Same — not a tabular format. | Ignore — model files don't need preview. |
| Preview hangs forever | Stale browser AAD token | Hard refresh (Cmd+Shift+R) or sign out / sign in. |
| Preview works in incognito, fails in your normal window | Browser extension / cached service worker | Clear `ml.azure.com` site data. |

**Rule of thumb**: AML Studio's data preview is meant for tabular exploration.
For anything else (binary blobs, model weights, HuggingFace dataset folders,
arbitrary directories), use the smoke test above to confirm reachability and
skip the Studio preview.
